# gold_payment_funnel

**Source:** `silver.payment_events`  
**Grain:** one row per event_name (funnel step)  
**Merge key:** `event_name`  
**Lineage:** Silver → Gold (never Bronze directly)

The `event` column in silver.payment_events is a JSONB-originated string.  
`get_json_object(event, '$.event_name')` extracts the event name; falls back to raw `event` value.

In [ ]:
dbutils.widgets.removeAll()
dbutils.widgets.text("catalog", "ubereats_dev")

In [ ]:
catalog    = dbutils.widgets.get("catalog")
gold_table = f"{catalog}.gold.payment_funnel"
silver_src = f"{catalog}.silver.payment_events"

print(f"[gold] {gold_table}  ←  {silver_src}")

In [ ]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {gold_table} (
        event_name           STRING NOT NULL,
        event_count          LONG,
        unique_payment_count LONG,
        _computed_at         TIMESTAMP NOT NULL
    ) USING DELTA
    CLUSTER BY (event_name)
    TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')
""")
print(f"[gold] table {gold_table} ready")

In [ ]:
from pyspark.sql.functions import col, coalesce, get_json_object, count, countDistinct, current_timestamp

# fact: silver.payment_events
# event may be JSON string {"event_name": "...", ...} or a plain string event name
funnel_df = (
    spark.table(silver_src)
    .withColumn(
        "event_name",
        coalesce(get_json_object(col("event"), "$.event_name"), col("event")),
    )
    .filter(col("event_name").isNotNull())
    .groupBy("event_name")
    .agg(
        count("event_id").alias("event_count"),
        countDistinct("payment_id").alias("unique_payment_count"),
    )
    .withColumn("_computed_at", current_timestamp())
)

print(f"[gold] {funnel_df.count()} funnel steps")

In [ ]:
funnel_df.createOrReplaceTempView("gold_payment_funnel_batch")

spark.sql(f"""
    MERGE INTO {gold_table} AS t
    USING gold_payment_funnel_batch AS s
    ON t.event_name = s.event_name
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")

print(f"[gold] {gold_table} updated")